# 6. Model card and summary

This notebook does not train anything. It reads the results saved by notebooks 4 and 5 (`results/*.json`) so every number here comes from code that ran, and puts the findings, the limits, and what I would do next in one place.

In [1]:
import json
from pathlib import Path

import pandas as pd

RESULTS = Path.cwd().parent / "results"
reg = json.loads((RESULTS / "regression.json").read_text())
clf = json.loads((RESULTS / "classification.json").read_text())
pd.set_option("display.precision", 3)

## Model 1: hourly bike rentals (regression)

| | |
|---|---|
| **Purpose** | Estimate rentals per hour given the calendar and that hour's weather. |
| **Data** | UCI Bike Sharing, Capital Bikeshare, Washington DC, 2011 to 2012, 17,379 hours. |
| **Target** | `cnt`, modelled as `log1p(cnt)` and scored back on the count scale. |
| **Excluded** | `casual` and `registered` (they sum to the target), `feels_like_c` for linear models (VIF over 60 with `temp_c`). |
| **Validation** | Train before 2012-10-01, test on Oct to Dec 2012. Tuning with expanding-window `TimeSeriesSplit`, so no fold sees the future. |
| **Chosen model** | LightGBM regressor. |

In [2]:
print("train:", reg["train_period"], f"({reg['n_train']:,} hours)")
print("test: ", reg["test_period"], f"({reg['n_test']:,} hours)")
pd.DataFrame(reg["test_metrics"]).T[["rmse", "mae", "r2"]]

train: ['2011-01-01 00:00:00', '2012-09-30 23:00:00'] (15,211 hours)
test:  ['2012-10-01 00:00:00', '2012-12-31 23:00:00'] (2,168 hours)


,rmse,mae,r2
baseline (hour x workingday mean),119.589,79.950,0.648
ridge,70.084,43.405,0.879
lasso,70.384,44.242,0.878
lightgbm,66.660,42.200,0.891


**What works.** The tree model and the regularised linear models all beat the hour by working-day baseline by a wide margin. The daily shape (commute peaks vs weekend hump) is captured well.

**What does not.** Errors are largest in rain or snow, which is rare in training. Every test month is under-predicted on average because demand kept growing and the model only knows about growth through a year flag.

**Limitations.** It uses the actual weather of each hour. A real forecast would feed it forecast weather and be less accurate. Two years of data means it has seen each calendar month only twice.

## Model 2: term deposit subscription (classification)

| | |
|---|---|
| **Purpose** | Rank clients by how likely they are to subscribe, to decide who to call. |
| **Data** | UCI Bank Marketing (`bank-additional-full`), Portuguese bank, May 2008 to Nov 2010, 41,188 calls. |
| **Target** | `y` (subscribed), 11.27% positive. |
| **Excluded** | `duration`. Call length is only known after the call, when the outcome is already known, so it leaks the answer. `pdays` replaced by a `previously_contacted` flag. |
| **Validation** | Stratified 80/20 holdout, stratified 5-fold CV on the 80%. A time-ordered check at the end. |
| **Chosen model** | LightGBM classifier, unweighted, with a cost-based threshold. |

In [3]:
print(f"train {clf['n_train']:,} rows, test {clf['n_test']:,} rows, positive rate {clf['positive_rate']:.4f}")
print("\n5-fold CV (train split):")
display(pd.DataFrame(clf["cv_5fold"]).T[["roc_auc", "pr_auc", "brier"]])
print("Test split:")
display(pd.DataFrame(clf["test"]).T)

train 32,950 rows, test 8,238 rows, positive rate 0.1127

5-fold CV (train split):


,roc_auc,pr_auc,brier
logistic,0.789,0.450,0.079
logistic (balanced),0.790,0.444,0.165
random forest,0.792,0.469,0.077
lightgbm,0.795,0.469,0.077


Test split:


,roc_auc,pr_auc,brier
logistic,0.801,0.465,0.077
logistic (balanced),0.801,0.460,0.162
random forest,0.809,0.494,0.074
lightgbm,0.814,0.495,0.074


In [4]:
print(f"decision model: {clf['decision_model']}, cost of a wasted call = {clf['cost_fp']}, cost of a missed subscriber = {clf['cost_fn']}")
print(f"threshold chosen on out-of-fold training predictions: {clf['threshold']}")
pd.DataFrame(clf["policies"]).set_index("policy")

decision model: lightgbm, cost of a wasted call = 5, cost of a missed subscriber = 50
threshold chosen on out-of-fold training predictions: 0.11


,calls,subscribers reached,missed,precision,recall,total cost
policy,,,,,,
call nobody,0,0,928,NaN,0.000,46400.0
default threshold 0.5,407,268,660,0.658,0.289,33695.0
cost-based threshold 0.11,1627,615,313,0.378,0.663,20710.0
call everyone,8238,928,0,0.113,1.000,36550.0


In [5]:
print("top features by permutation importance:", ", ".join(clf["top_permutation_features"]))
print("\ntime-ordered check (train on first 80% of rows, test on last 20%):")
pd.DataFrame(clf["time_ordered_check"]).T

top features by permutation importance: nr_employed, euribor3m, month, previously_contacted, contact

time-ordered check (train on first 80% of rows, test on last 20%):


,roc_auc,pr_auc,brier,mean_predicted,actual_rate
logistic,0.748,0.527,0.191,0.256,0.308
lightgbm,0.658,0.445,0.250,0.093,0.308


**What works.** Probabilities from the unweighted models are well calibrated, so a threshold based on costs behaves the way theory says it should. At the stated costs that threshold cuts total cost a lot compared with the default 0.5 or calling everyone.

**What does not.** The gain from tree models over logistic regression is small on a random split, and it reverses on a time split. The strongest features are economy-wide indicators that describe when a call was made, not who the client is.

**Limitations.**
- The cost numbers (5 per wasted call, 50 per missed subscriber) are my assumption, not the bank's. The method carries over, the numbers do not.
- The random split overstates real-world performance. The time-ordered check shows a big drop for LightGBM, because the later period's interest rates and employment are outside the training range.
- This is observational data from one bank in one economic period. Nothing here says that calling someone on a mobile phone *causes* a subscription.
- Fairness: `age`, `marital` and `job` are in the feature set. Before using a model like this for real decisions I would check how its errors fall across age groups.

## Statistics, in one paragraph each

**Sampling (notebook 2).** On a skewed real population the sample mean was close to normal by n = 30, and both t and bootstrap intervals came out near 95% coverage.

**Testing (notebook 3).** Total daily rentals did not differ detectably between working and non-working days, but the test was underpowered for an effect that small. Casual riders, contact type and weather all showed clear effects. With 41,188 rows nearly every categorical feature was "significant", so effect sizes did more of the filtering than p-values did. The simulated A/B test hit its planned false positive rate and power, and peeking 10 times more than tripled the false positive rate.

## What I would do next

1. Bank: validate with a rolling time split and see whether a model without the macro columns is more stable.
2. Bank: build an uplift model if there were treatment and control groups, since the real question is who changes their mind because of the call.
3. Bike: add lagged demand and a trend term, and test with forecast weather instead of observed weather.